# 실험 07 — 공통 오분류 환자군의 유전체 특성 분석

**연구 질문:** 4개 모델이 공통으로 잘못 분류한 환자들은, 같은 암종에서 정답으로 분류된 환자와 유전체 변이 패턴이 다른가?

**대상:** `train(1).csv`에 해당하는 Train 6,201명, 실험 06의 3-Fold OOF(Out-of-Fold) 예측. **Test 데이터는 열지 않음.**

**연구 순서:** (A) 입력·예측 무결성 점검 → (B) 암종별 공통 실패 → (C) 변이 수와 클래스 구성 분리 → (D) 주요 오류 방향 → (E) 같은 암종 내 유전자 변이율 비교 → (F) 다음 실험 가설 수립.

> ⚠️ 실험 07은 **사후 탐색적 오류 분석**입니다. 검증 정답으로 환자군을 정의하고 유전자를 찾는 과정은 최종 제출용 특징 선택의 근거로 그대로 사용하면 선택 편향/데이터 누출 위험이 있습니다. 새로운 개선 전략은 학습 Fold 안에서만 특징을 선택하고 별도 검증(중첩 교차검증 또는 미사용 홀드아웃)해야 합니다.

## 0. 환경 / 경로 설정

기존 프로젝트의 `outputs/`에는 실험 01의 `X_train_binary.npz`, `y_train.npy`, `gene_columns.json`이 있어야 합니다. 실험 06 결과는 `experiment_06/` 폴더에 압축 해제하거나 ZIP 파일을 프로젝트 루트에 두면 자동 인식합니다.

필요한 라이브러리가 없다면 `%pip install numpy pandas scipy scikit-learn matplotlib` 실행 후 커널을 다시 시작하세요.

In [ ]:
from pathlib import Path
import io, json, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import mannwhitneyu, fisher_exact
from statsmodels.stats.multitest import multipletests
from IPython.display import display

ROOT = Path.cwd()
DATA_DIR = ROOT / 'outputs'
OUT_DIR = ROOT / 'experiment_07'
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXP06_DIR = ROOT / 'experiment_06'
EXP06_ZIP = ROOT / 'experiment_06.zip'

# 다운로드 노트북을 /mnt/data 등에 놓고 바로 실행할 경우에만 아래 경로를 조정하세요.
print('Working directory:', ROOT)
print('Output directory:', OUT_DIR.resolve())

## 1. 실험 06 예측 및 Train 변이 행렬 불러오기

In [ ]:
def load_exp06_csv(filename):
    path = EXP06_DIR / filename
    if path.exists():
        return pd.read_csv(path)
    if EXP06_ZIP.exists():
        with zipfile.ZipFile(EXP06_ZIP) as archive:
            names = [s for s in archive.namelist() if s.endswith('/'+filename) or s == filename]
            if len(names) != 1:
                raise FileNotFoundError(f'{filename} was not uniquely found in {EXP06_ZIP}')
            return pd.read_csv(io.BytesIO(archive.read(names[0])))
    raise FileNotFoundError(f'Provide {EXP06_DIR / filename} or {EXP06_ZIP}')

X = sparse.load_npz(DATA_DIR / 'X_train_binary.npz').tocsr()
X.sum_duplicates(); X.eliminate_zeros(); X.sort_indices()
y = np.load(DATA_DIR / 'y_train.npy', allow_pickle=True).astype(str)
with open(DATA_DIR / 'gene_columns.json', encoding='utf-8') as f:
    gene_info = json.load(f)
genes = gene_info if isinstance(gene_info, list) else gene_info.get('gene_columns', gene_info.get('genes'))
assert genes is not None, 'gene_columns.json 형태를 확인하세요.'
assert X.shape == (len(y), len(genes)), (X.shape, len(y),len(genes))
assert np.isin(X.data, [1]).all(), 'X must contain 0/1 binary values.'
oof = load_exp06_csv('oof_predictions.csv')
hard_saved = load_exp06_csv('hard_cases.csv')
print('X:', X.shape, 'OOF:', oof.shape, 'Models:', sorted(oof.model.unique()))

## 2. 행·라벨·모델 검증 및 공통 실패 그룹 재구성

- 모델당 모든 Train 행이 딱 한 번씩 검증되었는지 확인
- 4개 모델의 예측에 행 번호/정답/변이 수 차이가 없는지 확인
- 결과 CSV에 있던 `all_wrong`을 그대로 믿지 않고 **OOF 예측으로 재계산**

In [ ]:
models = sorted(oof['model'].unique().tolist())
assert len(models)==4, models
assert len(oof)==len(y)*len(models), (len(oof),len(y),len(models))
assert oof[['row_index','model']].duplicated().sum()==0
assert oof.row_index.between(0,len(y)-1).all()
assert oof.groupby('row_index')['fold'].nunique().eq(1).all()
assert np.array_equal(oof.sort_values(['model','row_index']).query('model == @models[0]')['row_index'].to_numpy(), np.arange(len(y)))
assert (oof['true'].to_numpy() == y[oof['row_index'].to_numpy()]).all(), 'OOF label alignment error'
assert (oof['correct'].astype(int).to_numpy() == (oof['true']==oof['pred']).astype(int).to_numpy()).all()
actual_burden=np.diff(X.indptr)
assert (oof['mutation_count'].to_numpy()==actual_burden[oof['row_index'].to_numpy()]).all()

wide = oof.pivot(index='row_index',columns='model',values='correct').reindex(range(len(y)))
assert wide.notna().all().all()
num_correct=wide.astype(int).sum(axis=1)
fold_by_row=oof.groupby('row_index')['fold'].first().reindex(range(len(y)))
patients = pd.DataFrame({
    'row_index':np.arange(len(y)), 'true':y, 'fold':fold_by_row.to_numpy(),
    'mutation_count':actual_burden, 'n_models_correct':num_correct.to_numpy()
})
patients['all_wrong']=patients.n_models_correct.eq(0)
patients['all_correct']=patients.n_models_correct.eq(4)
patients['case_group']=np.select([patients.all_wrong,patients.all_correct],['all_wrong','all_correct'],default='mixed')
assert (patients.all_wrong.to_numpy()==hard_saved.sort_values('row_index').all_wrong.to_numpy()).all()
print('Group counts:'); display(patients.case_group.value_counts().to_frame('n'))
print('Fold x case group:'); display(pd.crosstab(patients.fold,patients.case_group))

## 3. 암종별 공통 오분류 현황

환자 수, 공통 오분류율, 모든 모델 정답률을 암종별로 계산합니다. 단순 건수 순위만이 아니라 **비율**과 **표본 수**를 함께 확인하세요.

In [ ]:
class_summary=(patients.groupby('true',as_index=False)
    .agg(n=('row_index','size'),all_wrong_n=('all_wrong','sum'),all_correct_n=('all_correct','sum'),
         burden_median=('mutation_count','median')))
class_summary['all_wrong_rate']=class_summary.all_wrong_n/class_summary.n
class_summary['all_correct_rate']=class_summary.all_correct_n/class_summary.n
class_summary=class_summary.sort_values('all_wrong_rate',ascending=False)
display(class_summary.round(3))
class_summary.to_csv(OUT_DIR/'class_common_error_summary.csv',index=False,encoding='utf-8-sig')
fig,ax=plt.subplots(figsize=(12,5))
top=class_summary.head(15).iloc[::-1]
ax.barh(top['true'],top['all_wrong_rate'])
ax.set(xlabel='Common error rate (all 4 models wrong)',title='15 cancer labels with highest common error rates',xlim=(0,1))
plt.tight_layout(); plt.savefig(OUT_DIR/'class_common_errors.png',dpi=160); plt.show()

## 4. 변이 수와 공통 오분류의 관계

전반적 비교 외에 **같은 암종 내부의 공통 오분류군 vs 모두 정답군**을 비교합니다. 0개 환자는 동일한 이진 프로파일이어서 StratifiedGroupKFold에서 한 Fold에 묶일 수 있으므로 별도로 취급합니다.

In [ ]:
patients['burden_group']=pd.cut(patients.mutation_count,[-1,0,7,27,np.inf],
    labels=['0개','1~7개','8~27개','28개 이상'])
burden=(patients.groupby('burden_group',observed=False)
    .agg(n=('row_index','size'),all_wrong_n=('all_wrong','sum'),all_correct_n=('all_correct','sum')))
burden['all_wrong_rate']=burden.all_wrong_n/burden.n
burden=burden.reset_index()
display(burden)
burden.to_csv(OUT_DIR/'burden_common_errors.csv',index=False,encoding='utf-8-sig')

# 변이 0개 레이블 분포 / 어느 Fold에 집중되는지 조사
zero=patients[patients.mutation_count.eq(0)]
print('Zero-variant patients:',len(zero))
display(pd.crosstab(zero.true,zero.fold))
zero.groupby(['true','case_group'],as_index=False).size().to_csv(
    OUT_DIR/'zero_mutation_label_groups.csv',index=False,encoding='utf-8-sig')

# 클래스 구성비 차이를 제거하기 위한 암종 동일 가중 평균 (각 암종 내부 0개 제외)
nonzero=patients[patients.mutation_count.gt(0)].copy()
byclass=(nonzero.groupby(['true','burden_group'],observed=True)
    .agg(n=('row_index','size'),errors=('all_wrong','sum')).reset_index())
byclass['error_rate']=byclass.errors/byclass.n
byclass.to_csv(OUT_DIR/'class_by_burden.csv',index=False,encoding='utf-8-sig')
# 최소 10명의 관측이 있는 암종×변이구간만 평균함
balanced=(byclass[byclass.n>=10].groupby('burden_group',observed=True)
    .agg(class_count=('true','nunique'),class_mean_error=('error_rate','mean')))
print('Macro-averaged within-cancer error rates, min 10 people/class/bin:')
display(balanced)

## 5. 같은 암종 내부: 공통 오분류군과 모두 정답군의 변이 수

- 각 암종에서 두 그룹이 최소 10명 이상일 때만 비교
- Mann–Whitney 양측 검정 + **암종 간 FDR 보정**
- 이 검정은 상관관계 검토이며 인과관계를 의미하지 않음

In [ ]:
burden_rows=[]
for cancer,part in patients.groupby('true'):
    bad=part.loc[part.all_wrong,'mutation_count'].to_numpy()
    good=part.loc[part.all_correct,'mutation_count'].to_numpy()
    if len(bad)<10 or len(good)<10:
        continue
    stat,pval=mannwhitneyu(bad,good,alternative='two-sided')
    burden_rows.append(dict(cancer=cancer,n_wrong=len(bad),n_correct=len(good),
      wrong_median=float(np.median(bad)),correct_median=float(np.median(good)),
      median_diff=float(np.median(bad)-np.median(good)),p_value=pval))
burden_stats=pd.DataFrame(burden_rows)
if len(burden_stats):
    burden_stats['q_value']=multipletests(burden_stats.p_value,method='fdr_bh')[1]
    burden_stats=burden_stats.sort_values('q_value')
display(burden_stats.round(4))
burden_stats.to_csv(OUT_DIR/'within_cancer_burden_tests.csv',index=False,encoding='utf-8-sig')

## 6. 공통 오분류의 예측 방향

각 환자에 대해 네 모델이 **동일한 잘못된 암종을 선택했는지**와 서로 다른 암종으로 분산되었는지 구분합니다. 오류 유형에 따라 다음 해결 방향(라벨 혼동 vs 높은 예측 불확실성)이 달라질 수 있습니다.

In [ ]:
pwide=oof.pivot(index='row_index',columns='model',values='pred').reindex(range(len(y)))
wrong=patients[patients.all_wrong].copy()
wrong_preds=pwide.loc[wrong.row_index]
wrong['n_distinct_wrong_labels']=wrong_preds.nunique(axis=1).to_numpy()
wrong['unanimous_wrong']=wrong.n_distinct_wrong_labels.eq(1)
wrong['unanimous_pred']=np.where(wrong.unanimous_wrong,wrong_preds.iloc[:,0].to_numpy(),None)
print('Distribution of number of distinct wrong labels:')
display(wrong.n_distinct_wrong_labels.value_counts().sort_index().to_frame('patients'))
consensus=(wrong[wrong.unanimous_wrong].groupby(['true','unanimous_pred'])
    .size().rename('n').reset_index().sort_values('n',ascending=False))
display(consensus.head(20))
wrong.to_csv(OUT_DIR/'common_error_patient_features.csv',index=False,encoding='utf-8-sig')
consensus.to_csv(OUT_DIR/'unanimous_wrong_confusions.csv',index=False,encoding='utf-8-sig')

## 7. 같은 암종 내부: 공통 오분류군의 유전자 변이율 차이

**탐색용 후보 유전자**만 찾습니다. 관심 암종은 실험 04의 사전 정의된 주요 혼동 암종(`KIPAN,KIRC,GBMLGG,LGG,BRCA,OV`)과 소수 암종(`DLBC,ACC`)입니다.

- 각 암종 안에서 `all_wrong`과 `all_correct`를 비교하며, 중간(`mixed`) 그룹은 포함하지 않음
- 각 암종에서 합산 변이 환자 수가 **5명 이상인 유전자**만 Fisher exact test
- 유전자 검정은 암종별 Benjamini–Hochberg FDR 보정
- 유의하지 않은 유전자는 **효과 부재**가 아니라 **검정력 부족**일 수도 있음

**주의:** 이런 유전자는 검증 데이터의 정답을 사용해 발견한 **사후 발견**이므로, 이를 현재 폴드의 예측 모델에 넣고 같은 폴드로 평가하면 누출이 발생합니다. 후속 실험에서 특징 후보 선정은 학습 폴드 내부로 옮기세요.

In [ ]:
FOCUS_CLASSES=['KIPAN','KIRC','GBMLGG','LGG','BRCA','OV','DLBC','ACC']
gene_results=[]
for cancer in FOCUS_CLASSES:
    psub=patients[patients.true.eq(cancer)]
    bad_idx=psub.loc[psub.all_wrong,'row_index'].to_numpy(dtype=int)
    good_idx=psub.loc[psub.all_correct,'row_index'].to_numpy(dtype=int)
    if min(len(bad_idx),len(good_idx))<8:
        print(f'Skip {cancer}: group sizes {len(bad_idx)} / {len(good_idx)}')
        continue
    a=np.asarray(X[bad_idx].sum(axis=0)).ravel().astype(int)
    b=np.asarray(X[good_idx].sum(axis=0)).ravel().astype(int)
    candidate=np.where((a+b)>=5)[0]
    subresults=[]
    for j in candidate:
        _,pval=fisher_exact([[int(a[j]),len(bad_idx)-int(a[j])],
                             [int(b[j]),len(good_idx)-int(b[j])]],alternative='two-sided')
        subresults.append(dict(cancer=cancer,gene=str(genes[j]),
            n_all_wrong=len(bad_idx),n_all_correct=len(good_idx),
            n_mut_wrong=int(a[j]),n_mut_correct=int(b[j]),
            rate_wrong=float(a[j]/len(bad_idx)),rate_correct=float(b[j]/len(good_idx)),
            rate_diff=float(a[j]/len(bad_idx)-b[j]/len(good_idx)),p_value=float(pval)))
    if subresults:
        q=multipletests([r['p_value'] for r in subresults],method='fdr_bh')[1]
        for r,qv in zip(subresults,q):r['q_value']=float(qv)
        gene_results.extend(subresults)
        print(cancer,'tested genes:',len(subresults),'q<0.05:',sum(q<0.05))
gene_stats=pd.DataFrame(gene_results)
if len(gene_stats):
    gene_stats=gene_stats.sort_values(['cancer','q_value','p_value'])
    display(gene_stats.groupby('cancer',group_keys=False).head(5).round(4))
    gene_stats.to_csv(OUT_DIR/'within_cancer_gene_tests.csv',index=False,encoding='utf-8-sig')
else:
    print('No eligible gene tests.');pd.DataFrame().to_csv(OUT_DIR/'within_cancer_gene_tests.csv',index=False)

## 8. 핵심 검증 결과 요약 + 다음 실험(08) 가설 초안

이 셀은 **수치를 자동 생성해 저장**합니다. 단정적으로 인과관계를 주장하지 않고, 개선 전략의 테스트 대상을 구체화합니다.

**실험 08 설계 제안**

1. 공통 오분류 환자가 집중된 암종을 사전 지정하고, **동일 모델에서 학습 Fold 내부 특징 선택**을 적용한 모델과 기준 모델 비교
2. 변이 0개 환자는 입력 정보가 똑같다는 구조적 한계로 별도 분류·불확실성 검토 대상으로 취급하되 **대회 제출은 반드시 26개 클래스 중 하나를 예측**
3. 혼동 암종(예: KIPAN↔KIRC)에는 학습 폴드 안에서 선택한 특징만으로 2단계 분류기 또는 클래스별 보정 전략을 실험
4. 개선 전략을 선택하는 단계는 **중첩 CV 또는 아직 사용하지 않은 검증 분할**로 평가. 과거 OOF 결과에서 골라낸 유전자/그룹 기준을 같은 OOF로 다시 평가하지 않음.

In [ ]:
summary={
  'n_patients':int(len(patients)),
  'models':models,
  'all_wrong_n':int(patients.all_wrong.sum()),
  'all_correct_n':int(patients.all_correct.sum()),
  'all_wrong_fraction':float(patients.all_wrong.mean()),
  'zero_mutation_n':int((patients.mutation_count==0).sum()),
  'zero_mutation_all_wrong_n':int((patients.mutation_count.eq(0)&patients.all_wrong).sum()),
  'unanimous_wrong_n':int(wrong.unanimous_wrong.sum()),
  'notes':[
    'Only internal out-of-fold predictions and training features were used.',
    'Exploratory associations; avoid post-hoc feature selection on the same validation predictions.',
    'Zero-mutation group has identical binary input and may be concentrated in one validation fold.'
  ]
}
with open(OUT_DIR/'analysis_summary.json','w',encoding='utf-8') as f:
    json.dump(summary,f,ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('Saved files:',*[p.name for p in sorted(OUT_DIR.iterdir())],sep='\n  ')